# **Homework 4 - Generative Adversarial Network**

In this homework, you are required to build a generative adversarial  network for anime face generation.


**Name:** Ben Blake

**Student ID:** 14387365

## Set up the environment


### Packages Installation

In [19]:
# You may replace the workspace directory if you want.
workspace_dir = '.'

# Training progress bar
!pip install -q qqdm

### Mount the drive and unzip the dataset

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
!cp "/content/drive/My Drive/UMKC/Fall 2026/Deep Learning/HW4/HW4_Dataset.zip" "{workspace_dir}/HW4_Dataset.zip"

cp: cannot stat '/content/drive/MyDrive/Dataset.zip': No such file or directory


###Unzip the downloaded file.
The unzipped tree structure is like
```
faces/
├── 1.jpg
├── 2.jpg
├── 3.jpg
...
```

In [22]:
!unzip -q "{workspace_dir}/HW4_Dataset.zip" -d "{workspace_dir}/"

unzip:  cannot find or open ./crypko_data.zip, ./crypko_data.zip.zip or ./crypko_data.zip.ZIP.


## Random seed (1pt)
Set the random seed to a certain value for reproducibility.

In [23]:
import random

import torch
import numpy as np


def same_seeds(seed):
    # Python built-in random module
    random.seed(seed)
    # Numpy
    np.random.seed(seed)
    # Torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

same_seeds(6252002) # (1pt)TODO: Put your birthday as seed (MMDDYYYY)
# birthday is 06252002, but python 3 doesn't allow leading zeros on an int literal (06252002 is a
# SyntaxError), so it's written as 6252002, which is the same number

SyntaxError: invalid syntax (3681007248.py, line 20)

## Import Packages
First, we need to import packages that will be used later.

Like previous homeworks, we highly rely on **torchvision**, a library of PyTorch.

In [ ]:
import os
import glob

import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch import optim
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from qqdm.notebook import qqdm

## Dataset (2pts)
1. Resize the images to (64, 64)
1. Linearly map the values from [0, 1] to  [-1, 1].

Please refer to [PyTorch official website](https://pytorch.org/vision/stable/transforms.html) for details about different transforms.


In [ ]:
class CrypkoDataset(Dataset):
    def __init__(self, fnames, transform):
        self.transform = transform
        self.fnames = fnames
        self.num_samples = len(self.fnames)

    def __getitem__(self,idx):
        fname = self.fnames[idx]
        # (1pt)TODO: Load the image
        img = torchvision.io.read_image(fname) # hints:Using torchvision.io.read_image
        # reads the jpg into a uint8 tensor of shape (3, 96, 96) with values 0-255. every image in the set is 96x96 RGB
        # (1pt)TODO: Apply input transform
        img = self.transform(img) # hints: Only need use input transform above
        # runs the compose from get_dataset: ToPILImage, resize to 64x64, ToTensor (scales to [0, 1]),
        # then Normalize with mean and std 0.5, which maps [0, 1] to [-1, 1] to match the generator's tanh output
        return img

    def __len__(self):
        return self.num_samples


def get_dataset(root):
    fnames = glob.glob(os.path.join(root, '*'))
    # 1. Resize the image to (64, 64)
    # 2. Linearly map [0, 1] to [-1, 1]
    compose = [
        transforms.ToPILImage(),
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
    transform = transforms.Compose(compose)
    dataset = CrypkoDataset(fnames, transform)
    return dataset

### Show some images
Note that the values are in the range of [-1, 1], we should shift them to the valid range, [0, 1], to display correctly.

In [ ]:
dataset = get_dataset(os.path.join(workspace_dir, 'faces'))

images = [dataset[i] for i in range(16)]
grid_img = torchvision.utils.make_grid(images, nrow=4)
plt.figure(figsize=(10,10))
plt.imshow(grid_img.permute(1, 2, 0))
plt.show()

In [ ]:
images = [(dataset[i]+1)/2 for i in range(16)]
grid_img = torchvision.utils.make_grid(images, nrow=4)
plt.figure(figsize=(10,10))
plt.imshow(grid_img.permute(1, 2, 0))
plt.show()

## Model (10pts)
Here, we use DCGAN as the model structure. Feel free to modify your own model structure.

Note that the `N` of the input/output shape stands for the batch size.

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        m.weight.data.normal_(0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        m.weight.data.normal_(1.0, 0.02)
        m.bias.data.fill_(0)

class Generator(nn.Module):
    """
    Input shape: (N, in_dim)
    Output shape: (N, 3, 64, 64)
    """
    def __init__(self, in_dim, dim=64):
        super(Generator, self).__init__()
        def dconv_bn_relu(in_dim, out_dim):
            return nn.Sequential(
                # (1pt)TODO: Use nn.ConvTranspose2d
                nn.ConvTranspose2d(in_dim, out_dim, kernel_size=5, stride=2, padding=2, output_padding=1, bias=False), # hints: in_dim=in_dim, out_dim=out_dim, kernel_size=5, stride=2, padding=2, out_padding=1, bias=False
                # the hint's argument names are wrong: ConvTranspose2d takes in_channels/out_channels and
                # output_padding, not in_dim/out_dim/out_padding, so the channels are passed positionally.
                # with these settings each block exactly doubles H and W: (H-1)*2 - 2*2 + 5 + 1 = 2H.
                # no bias because the batchnorm right after it has its own shift term
                # (1pt)TODO: Batchnorm2d layer
                nn.BatchNorm2d(out_dim), #hints: dim=out_dim
                # one mean/var per channel, computed over the batch and both spatial dims
                # (1pt)TODO: ReLu layer
                nn.ReLU()
                # plain ReLU in the generator, like the DCGAN paper. the discriminator uses LeakyReLU instead
            )
        self.l1 = nn.Sequential(
            # (1pt)TODO: Linear layer
            nn.Linear(in_dim, dim * 8 * 4 * 4), # hints: in_dim=in_dim, out_dim=dim*8*4*4
            # projects z (100 dims) up to 64*8*4*4 = 8192, which forward() reshapes into 512 channels of 4x4
            # (1pt)TODO: Batchnorm1d layer
            nn.BatchNorm1d(dim * 8 * 4 * 4), # hints: dim=out dimension from the Linear layers
            # 1d instead of 2d because the Linear output is still flat (N, 8192) at this point
            # (1pt)TODO: ReLu layer
            nn.ReLU()
        )
        self.l2_5 = nn.Sequential(
            dconv_bn_relu(dim * 8, dim * 4),
            dconv_bn_relu(dim * 4, dim * 2),
            dconv_bn_relu(dim * 2, dim),
            nn.ConvTranspose2d(dim, 3, 5, 2, padding=2, output_padding=1),
            # (1pt)TODO:Tanh Layer
            nn.Tanh()
            # squashes the output to [-1, 1], the same range the dataset's Normalize puts real images in.
            # spatially it's gone 4 -> 8 -> 16 -> 32 -> 64, so the output is (N, 3, 64, 64)
        )
        self.apply(weights_init)

    def forward(self, x):
        y = self.l1(x)
        y = y.view(y.size(0), -1, 4, 4)
        y = self.l2_5(y)
        return y


class Discriminator(nn.Module):
    """
    Input shape: (N, 3, 64, 64)
    Output shape: (N, )
    """
    def __init__(self, in_dim, dim=64):
        super(Discriminator, self).__init__()

        def conv_bn_lrelu(in_dim, out_dim):
            return nn.Sequential(
                # (1pt)TODO: Conv layer
                nn.Conv2d(in_dim, out_dim, 5, 2, 2), #hints: Conv2d layer with in_dim,out_dim,kernel=5,stride and padding=2
                # the hint leaves out the stride value. it has to be 2, same as the first conv in self.ls:
                # each block halves H and W (floor((H + 2*2 - 5)/2) + 1 = H/2), so 64 -> 32 -> 16 -> 8 -> 4,
                # and the last Conv2d(dim*8, 1, 4) turns that 4x4 into a single score per image.
                # with stride 1 the map would still be 32x32 there and D wouldn't output shape (N,)
                nn.BatchNorm2d(out_dim),
                # (1pt)TODO: Leaky ReLu layer with 0.2
                nn.LeakyReLU(0.2),
                # slope 0.2 for negative inputs, matching the LeakyReLU after the first conv
            )

        self.ls = nn.Sequential(
            nn.Conv2d(in_dim, dim, 5, 2, 2),
            nn.LeakyReLU(0.2),
            conv_bn_lrelu(dim, dim * 2),
            conv_bn_lrelu(dim * 2, dim * 4),
            conv_bn_lrelu(dim * 4, dim * 8),
            nn.Conv2d(dim * 8, 1, 4),
            # (1pt)TODO: Sigmoid layer
            nn.Sigmoid(),
            # maps the score to a probability in (0, 1) that the image is real, which is what BCELoss expects
        )
        self.apply(weights_init)

    def forward(self, x):
        y = self.ls(x)
        y = y.view(-1)
        return y

## Training

### Initialization (3pts)
- hyperparameters
- model
- optimizer
- dataloader

In [ ]:
# Training hyperparameters
batch_size = 64
z_dim = 100
z_sample = Variable(torch.randn(100, z_dim)).cuda()
lr = 1e-4

# You many try other number of epochs to see the performance
# Every epoch will take 10 mins for training
n_epoch = 6 # 20
n_critic = 1 # 2
# clip_value = 0.01

log_dir = os.path.join(workspace_dir, 'logs')
ckpt_dir = os.path.join(workspace_dir, 'checkpoints')
os.makedirs(log_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

# Model
G = Generator(in_dim=z_dim).cuda()
D = Discriminator(3).cuda()
G.train()
D.train()

# Loss
# (1pt)TODO: Binary Cross Entropy loss
criterion = nn.BCELoss()
# plain BCELoss rather than BCEWithLogitsLoss, since D already ends in a Sigmoid. the loop trains D
# toward 1 for real and 0 for fake, and trains G by scoring its fakes against the real label (1)

# Optimizer
# (1pt)TODO: Adam Optimizer
opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999)) # hints: D.parameters()
# lr is the 1e-4 set above. betas=(0.5, 0.999) is what the DCGAN paper uses; it found the
# default beta1 of 0.9 made training oscillate and go unstable
# (1pt)TODO: Adam optimizer
opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999)) # hints: G.parameters()
# same settings for G. each optimizer only holds its own model's parameters, so opt_D.step()
# never moves G's weights and opt_G.step() never moves D's
# opt_D = torch.optim.RMSprop(D.parameters(), lr=lr)
# opt_G = torch.optim.RMSprop(G.parameters(), lr=lr)


# DataLoader
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)


### Training loop
We store some pictures regularly to monitor the current performance of the Generator, and regularly record checkpoints.

In [ ]:
steps = 0
for e, epoch in enumerate(range(n_epoch)):
    progress_bar = qqdm(dataloader)
    for i, data in enumerate(progress_bar):
        imgs = data
        imgs = imgs.cuda()

        bs = imgs.size(0)

        # ============================================
        #  Train D
        # ============================================
        z = Variable(torch.randn(bs, z_dim)).cuda()
        r_imgs = Variable(imgs).cuda()
        f_imgs = G(z)

        # Label
        r_label = torch.ones((bs)).cuda()
        f_label = torch.zeros((bs)).cuda()

        # Model forwarding
        r_logit = D(r_imgs.detach())
        f_logit = D(f_imgs.detach())

        # Compute the loss for the discriminator.
        r_loss = criterion(r_logit, r_label)
        f_loss = criterion(f_logit, f_label)
        loss_D = (r_loss + f_loss) / 2

        # WGAN Loss
        # loss_D = -torch.mean(D(r_imgs)) + torch.mean(D(f_imgs))


        # Model backwarding
        D.zero_grad()
        loss_D.backward()

        # Update the discriminator.
        opt_D.step()

        # for p in D.parameters():
        #    p.data.clamp_(-clip_value, clip_value)

        # ============================================
        #  Train G
        # ============================================
        if steps % n_critic == 0:
            # Generate some fake images.
            z = Variable(torch.randn(bs, z_dim)).cuda()
            f_imgs = G(z)

            # Model forwarding
            f_logit = D(f_imgs)

            # Compute the loss for the generator.
            loss_G = criterion(f_logit, r_label)
            # WGAN Loss
            # loss_G = -torch.mean(D(f_imgs))

            # Model backwarding
            G.zero_grad()
            loss_G.backward()

            # Update the generator.
            opt_G.step()

        steps += 1

        # Set the info of the progress bar
        #   Note that the value of the GAN loss is not directly related to
        #   the quality of the generated images.
        progress_bar.set_infos({
            'Loss_D': round(loss_D.item(), 4),
            'Loss_G': round(loss_G.item(), 4),
            'Epoch': e+1,
            'Step': steps,
        })

    G.eval()
    f_imgs_sample = (G(z_sample).data + 1) / 2.0
    filename = os.path.join(log_dir, f'Epoch_{epoch+1:03d}.jpg')
    torchvision.utils.save_image(f_imgs_sample, filename, nrow=10)
    print(f' | Save some samples to {filename}.')

    # Show generated images in the jupyter notebook.
    grid_img = torchvision.utils.make_grid(f_imgs_sample.cpu(), nrow=10)
    plt.figure(figsize=(10,10))
    plt.imshow(grid_img.permute(1, 2, 0))
    plt.show()
    G.train()

    if (e+1) % 5 == 0 or e == 0:
        # Save the checkpoints.
        torch.save(G.state_dict(), os.path.join(ckpt_dir, 'G.pth'))
        torch.save(D.state_dict(), os.path.join(ckpt_dir, 'D.pth'))



## Inference
Use the trained model to generate anime faces!

### Load model

In [ ]:
import torch

G = Generator(z_dim)
G.load_state_dict(torch.load(os.path.join(ckpt_dir, 'G.pth')))
G.eval()
G.cuda()

### Generate and show some images.


In [ ]:
# Generate 1000 images and make a grid to save them.
n_output = 1000
z_sample = Variable(torch.randn(n_output, z_dim)).cuda()
imgs_sample = (G(z_sample).data + 1) / 2.0
log_dir = os.path.join(workspace_dir, 'logs')
filename = os.path.join(log_dir, 'result.jpg')
torchvision.utils.save_image(imgs_sample, filename, nrow=10)

# Show 32 of the images.
grid_img = torchvision.utils.make_grid(imgs_sample[:32].cpu(), nrow=10)
plt.figure(figsize=(10,10))
plt.imshow(grid_img.permute(1, 2, 0))
plt.show()